<div style="background: linear-gradient(135deg, #1a2535 0%, #2d3f55 100%);
     padding: 40px 30px; border-radius: 12px; margin-bottom: 10px; border-left: 6px solid #4a9eff;">

<h1 style="color: #ffffff; font-size: 2em; margin: 0 0 8px 0;">
  📦 Data Mining sur un Data Warehouse
</h1>
<h2 style="color: #4a9eff; font-size: 1.3em; margin: 0 0 20px 0; font-weight: normal;">
  Règles d'association — Algorithme Apriori
</h2>

<table style="color: #ccd6f6; font-size: 0.9em; border-collapse: collapse;">
  <tr><td style="padding: 3px 20px 3px 0;"><b>Cours</b></td><td>Business Intelligence — HES-SO Valais</td></tr>
  <tr><td style="padding: 3px 20px 3px 0;"><b>Dataset</b></td><td>SwissShop — 3 000 transactions synthétiques réalistes</td></tr>
  <tr><td style="padding: 3px 20px 3px 0;"><b>Méthode</b></td><td>Protocole CRISP-DM | Algorithme Apriori (mlxtend)</td></tr>
  <tr><td style="padding: 3px 20px 3px 0;"><b>Durée estimée</b></td><td>3 × 45 min</td></tr>
</table>
</div>

---

## 🎯 Objectifs d'apprentissage

À la fin de ce notebook, vous serez capables de :

1. **Comprendre** le problème du *Market Basket Analysis* et sa valeur business
2. **Maîtriser** les trois métriques fondamentales : Support, Confiance, Lift
3. **Implémenter** l'algorithme Apriori sur un dataset réel avec Python
4. **Interpréter** les règles extraites et les traduire en décisions business concrètes
5. **Visualiser** les résultats de façon professionnelle

---

## 📋 Plan du notebook

| # | Section | Contenu |
|---|---------|---------|
| 1 | **Contexte & Théorie** | Problème, métriques, algorithme |
| 2 | **Environnement** | Imports, configuration |
| 3 | **Dataset SwissShop** | Chargement, exploration |
| 4 | **Preprocessing** | Format basket, nettoyage |
| 5 | **Algorithme Apriori** | Itemsets fréquents |
| 6 | **Règles d'association** | Génération, filtrage |
| 7 | **Visualisations** | Heatmap, réseau, distributions |
| 8 | **Règles Business** | Traduction, recommandations |
| 9 | **Exercices** | Mise en pratique autonome |
| 10 | **Synthèse** | Points-clés, limites, CRISP-DM |


---
# 📚 Section 1 — Contexte & Théorie

## 1.1 Le problème du Market Basket Analysis

<div style="background: #f8f4e8; border-left: 4px solid #e6a817;
     padding: 15px 20px; border-radius: 6px; margin: 15px 0;">
<b>🛒 Situation :</b> Vous êtes analyste BI chez SwissShop, une chaîne de distribution
tech en Suisse. Le directeur commercial vous demande :<br><br>
<i>« Quels produits nos clients achètent-ils ensemble, et comment exploiter ça ? »</i>
</div>

L'analyse des paniers d'achat (*Market Basket Analysis*) répond précisément à cette question.
Elle permet de découvrir des **patterns de co-occurrence** dans les données transactionnelles
et d'en déduire des **règles du type** :

```
SI un client achète {Laptop Pro 15", Souris sans fil}
ALORS il achètera probablement {Hub USB-C}
```

### Applications business directes

- **Cross-selling** : proposer automatiquement un produit complémentaire
- **Bundling** : créer des offres groupées à prix attractif
- **Placement en rayon** : rapprocher physiquement les produits associés
- **Recommandations e-commerce** : "Les clients qui ont acheté X ont aussi acheté Y"
- **Gestion des stocks** : anticiper les ruptures groupées

---

## 1.2 Les trois métriques fondamentales

### 🔵 Support — *Quelle est la fréquence du phénomène ?*

$$\text{Support}(A \rightarrow B) = \frac{|\text{transactions contenant A et B}|}{|\text{total transactions}|}$$

**Exemple :** Si 150 transactions sur 3 000 contiennent à la fois *Laptop* et *Souris* :
$$\text{Support} = \frac{150}{3000} = 0{,}05 = 5\%$$

> ⚠️ Un support faible n'invalide pas la règle — certains produits chers sont rarement achetés mais très fortement associés.

---

### 🟡 Confiance — *Quelle est la force directionnelle de la règle ?*

$$\text{Confidence}(A \rightarrow B) = \frac{\text{Support}(A \cup B)}{\text{Support}(A)}$$

**Exemple :** Si parmi les acheteurs de *Laptop*, 73% achètent aussi une *Souris* :
$$\text{Confidence} = 0{,}73 = 73\%$$

> ⚠️ La confiance est **asymétrique** : Confidence(A→B) ≠ Confidence(B→A)

---

### 🟢 Lift — *La règle est-elle meilleure que le hasard ?*

$$\text{Lift}(A \rightarrow B) = \frac{\text{Confidence}(A \rightarrow B)}{\text{Support}(B)}$$

| Lift | Interprétation |
|------|---------------|
| = 1 | Indépendance — A et B sont achetés indépendamment |
| > 1 | Corrélation **positive** — A favorise l'achat de B ✅ |
| < 1 | Corrélation **négative** — A réduit la probabilité d'acheter B |

> 🎯 **Règle pratique :** Lift ≥ 1.5 = règle exploitable. Lift ≥ 3.0 = règle forte.

---

## 1.3 L'algorithme Apriori — Principe

### Le principe d'anti-monotonicité

<div style="background: #e8f4f8; border-left: 4px solid #4a9eff;
     padding: 15px 20px; border-radius: 6px; margin: 15px 0;">
<b>Propriété fondamentale :</b> Si un itemset est <i>peu fréquent</i>,
<b>tous ses sur-ensembles</b> le sont aussi — et peuvent donc être <b>ignorés</b>.
</div>

```
Exemple de pruning :
─────────────────────────────────────────────────────
{Laptop}               → support = 12%  ✅ fréquent
{Laptop, Souris}       → support = 8%   ✅ fréquent
{Laptop, Souris, Hub}  → support = 4%   ✅ fréquent
{Laptop, Clavier}      → support = 0.3% ❌ sous le seuil
   └→ {Laptop, Clavier, *} = IGNORÉ     🚫 pruning
─────────────────────────────────────────────────────
```

Ce mécanisme réduit **drastiquement** l'espace de recherche combinatoire.

### Les étapes

```
1. Générer les itemsets fréquents de taille 1 (L1) — filtrer par min_support
2. Combiner L1 → itemsets de taille 2 (L2) — filtrer
3. Combiner L2 → L3 … répéter jusqu'à épuisement
4. À partir de tous les Lk, générer les règles A → B
5. Filtrer les règles par min_confidence et min_lift
```


---
# 🛠️ Section 2 — Environnement & Imports

## 2.1 Installation des dépendances

Exécutez la cellule suivante **une seule fois** pour installer les librairies nécessaires.


In [ ]:
# ── Installation des librairies (à exécuter une seule fois) ──────────────────
# Décommentez si nécessaire :
# !pip install mlxtend pandas numpy matplotlib seaborn networkx

## 2.2 Imports & Configuration

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import warnings

from mlxtend.frequent_patterns import apriori, association_rules

warnings.filterwarnings('ignore')

# ── Style global des graphiques ───────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1923',
    'axes.facecolor':   '#1a2535',
    'axes.edgecolor':   '#3a4a5a',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'text.color':       'white',
    'grid.color':       '#2a3a4a',
    'grid.alpha':       0.4,
})

# ── Seed pour la reproductibilité ─────────────────────────────────────────────
np.random.seed(42)

print("✅ Environnement prêt")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")
import mlxtend; print(f"   mlxtend {mlxtend.__version__}")

---
# 📦 Section 3 — Dataset SwissShop

## 3.1 Génération du dataset synthétique

Le dataset SwissShop simule une boutique tech suisse. Il a été construit avec des
**patterns d'achat réalistes** injectés volontairement — ce qui nous permettra de
vérifier que notre algorithme les retrouve.

<div style="background: #f0f8e8; border-left: 4px solid #6bcb77;
     padding: 12px 18px; border-radius: 6px; margin: 10px 0; color: #1a3a1a;">
<b>📌 Note pédagogique :</b> En situation réelle, ce dataset proviendrait d'une
requête SQL sur le Data Warehouse (table <code>fact_sales</code> + <code>dim_product</code>).
La structure est identique à ce que vous trouveriez dans un DWH en étoile.
</div>


In [ ]:
# ══════════════════════════════════════════════════════════════════
# GÉNÉRATION DU DATASET SWISSSHOP
# ══════════════════════════════════════════════════════════════════

# ── Catalogue produits ────────────────────────────────────────────
CATALOGUE = {
    "Informatique": [
        "Laptop Pro 15\"", "Laptop Ultrabook", "Souris sans fil",
        "Clavier mécanique", "Clavier sans fil", "Webcam HD",
        "Hub USB-C", "Disque SSD externe", "Écran 27\" 4K", "Casque Bluetooth"
    ],
    "Téléphonie": [
        "Smartphone Android", "Smartphone iPhone", "Coque protection",
        "Chargeur rapide", "Câble USB-C", "Protection écran", "Batterie externe"
    ],
    "Audio": [
        "Écouteurs True Wireless", "Enceinte Bluetooth", "Casque HiFi",
        "Adaptateur Jack", "Câble Audio optique"
    ],
    "Bureau": [
        "Imprimante laser", "Cartouche toner", "Papier A4 500f",
        "Lampe de bureau LED", "Support laptop", "Tapis de souris XL"
    ],
    "Accessoires": [
        "Sac à dos PC", "Housse laptop 15\"", "Multiprise USB",
        "Nettoyant écran", "Câble HDMI 2.1", "Lecteur carte SD"
    ]
}

ALL_PRODUCTS = [p for cat in CATALOGUE.values() for p in cat]

# ── Patterns d'achat réalistes ────────────────────────────────────
PATTERNS = [
    (["Laptop Pro 15\""],         ["Souris sans fil", "Hub USB-C"],          0.70),
    (["Laptop Pro 15\""],         ["Clavier mécanique"],                     0.45),
    (["Laptop Pro 15\""],         ["Sac à dos PC"],                          0.55),
    (["Laptop Ultrabook"],        ["Housse laptop 15\"", "Souris sans fil"],  0.65),
    (["Laptop Ultrabook"],        ["Hub USB-C"],                              0.60),
    (["Écran 27\" 4K"],           ["Câble HDMI 2.1"],                        0.85),
    (["Écran 27\" 4K"],           ["Clavier sans fil", "Souris sans fil"],   0.50),
    (["Smartphone Android"],      ["Coque protection", "Protection écran"],   0.75),
    (["Smartphone Android"],      ["Chargeur rapide"],                        0.60),
    (["Smartphone iPhone"],       ["Coque protection"],                       0.80),
    (["Smartphone iPhone"],       ["Protection écran", "Câble USB-C"],        0.65),
    (["Chargeur rapide"],         ["Câble USB-C"],                            0.78),
    (["Imprimante laser"],        ["Cartouche toner", "Papier A4 500f"],      0.90),
    (["Imprimante laser"],        ["Câble USB-C"],                            0.55),
    (["Souris sans fil","Clavier sans fil"], ["Tapis de souris XL"],          0.55),
    (["Laptop Pro 15\"","Écran 27\" 4K"],   ["Support laptop"],              0.60),
    (["Disque SSD externe"],      ["Câble USB-C"],                            0.70),
    (["Webcam HD"],               ["Casque Bluetooth"],                       0.45),
    (["Batterie externe"],        ["Câble USB-C"],                            0.80),
]

def generate_transaction():
    basket = set()
    start = np.random.choice(ALL_PRODUCTS)
    basket.add(start)
    for triggers, associated, prob in PATTERNS:
        if any(t in basket for t in triggers):
            for product in associated:
                if np.random.random() < prob:
                    basket.add(product)
    n_extra = np.random.choice([0, 1, 2], p=[0.60, 0.30, 0.10])
    for _ in range(n_extra):
        basket.add(np.random.choice(ALL_PRODUCTS))
    return list(basket)

# ── Génération des transactions ───────────────────────────────────
N_TRANSACTIONS = 3000
transactions_list = [generate_transaction() for _ in range(N_TRANSACTIONS)]

# ── Construction du DataFrame ─────────────────────────────────────
records = []
for tid, items in enumerate(transactions_list, 1):
    date = pd.Timestamp('2024-01-01') + pd.Timedelta(days=np.random.randint(0, 365))
    customer = f"C{np.random.randint(1000, 9999)}"
    for item in items:
        cat = next((c for c, p in CATALOGUE.items() if item in p), "Autre")
        records.append({
            "transaction_id": f"T{tid:05d}",
            "customer_id":    customer,
            "date":           date,
            "product_name":   item,
            "category":       cat,
            "price_chf":      round(np.random.uniform(9.90, 1299.00), 2)
        })

df = pd.DataFrame(records)

print("✅ Dataset SwissShop généré")
print(f"   Transactions  : {df['transaction_id'].nunique():,}")
print(f"   Lignes totales: {len(df):,}")
print(f"   Produits      : {df['product_name'].nunique()}")
print(f"   Catégories    : {df['category'].nunique()}")
print(f"   Période       : {df['date'].min().date()} → {df['date'].max().date()}")
print()
print("Aperçu des premières lignes :")
df.head(8)

## 3.2 Exploration des données

In [ ]:
# ── Statistiques descriptives ─────────────────────────────────────
print("=" * 50)
print("STATISTIQUES DESCRIPTIVES")
print("=" * 50)

# Taille des paniers
basket_sizes = df.groupby('transaction_id')['product_name'].count()
print(f"\nTaille moyenne d'un panier : {basket_sizes.mean():.2f} produits")
print(f"Taille min / max           : {basket_sizes.min()} / {basket_sizes.max()}")
print(f"Médiane                    : {basket_sizes.median():.0f} produits")

# Produits les plus vendus
print("\n Top 10 produits les plus fréquents :")
top_products = df['product_name'].value_counts().head(10)
for product, count in top_products.items():
    pct = count / N_TRANSACTIONS * 100
    bar = "█" * int(pct / 2)
    print(f"  {product:<30} {count:>5} ({pct:5.1f}%)  {bar}")

In [ ]:
# ── Visualisation : distribution par catégorie et taille des paniers ──────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Panneau gauche : ventes par catégorie
cat_counts = df['category'].value_counts()
colors_cat = ['#4a9eff', '#ff6b6b', '#ffd93d', '#6bcb77', '#c77dff']
bars = axes[0].barh(cat_counts.index, cat_counts.values,
                    color=colors_cat, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, cat_counts.values):
    axes[0].text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9)
axes[0].set_title("Lignes de vente par catégorie", fontsize=12, pad=10)
axes[0].set_xlabel("Nombre de lignes")
axes[0].set_xlim(0, cat_counts.max() * 1.15)

# Panneau droit : distribution de la taille des paniers
axes[1].hist(basket_sizes, bins=range(1, basket_sizes.max() + 2),
             color='#4a9eff', edgecolor='white', linewidth=0.5, alpha=0.8)
axes[1].axvline(basket_sizes.mean(), color='#ffd93d', linestyle='--',
                linewidth=2, label=f'Moyenne = {basket_sizes.mean():.1f}')
axes[1].set_title("Distribution de la taille des paniers", fontsize=12, pad=10)
axes[1].set_xlabel("Nombre de produits par transaction")
axes[1].set_ylabel("Fréquence")
axes[1].legend()

plt.suptitle("SwissShop — Exploration des données", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
# 🔧 Section 4 — Préparation des données (Preprocessing)

## 4.1 Transformation en format "basket"

L'algorithme Apriori nécessite une **matrice booléenne** :
- Chaque **ligne** = une transaction
- Chaque **colonne** = un produit
- Chaque **cellule** = `True` si le produit est dans la transaction, `False` sinon

```
transaction_id | Laptop | Souris | Hub USB-C | Câble USB-C | ...
T00001         |  True  |  True  |   True    |    False    | ...
T00002         | False  | False  |  False    |    True     | ...
```

<div style="background: #fef3e2; border-left: 4px solid #e6a817;
     padding: 12px 18px; border-radius: 6px; margin: 10px 0; color: #3a2a0a;">
<b>⚠️ Point critique :</b> On veut <b>présence/absence</b> (True/False),
pas la <b>quantité</b> achetée. Apriori travaille sur des ensembles, pas des multiensembles.
</div>


In [ ]:
# ── Transformation en matrice booléenne ──────────────────────────────────────
basket = (
    df.groupby(['transaction_id', 'product_name'])['product_name']
    .count()
    .unstack(fill_value=0)
    .astype(bool)
)

print(f"Dimensions de la matrice basket : {basket.shape}")
print(f"  → {basket.shape[0]:,} transactions × {basket.shape[1]} produits")
print(f"\nDensité de la matrice : {basket.values.sum() / basket.size:.2%}")
print("  (proportion de cellules True — naturellement sparse)")
print()
print("Aperçu de la matrice (5 premières transactions, 6 premiers produits) :")
basket.iloc[:5, :6]

## 4.2 Vérification de la qualité

Avant de lancer l'algorithme, vérifiez que :
- Il n'y a **pas de doublons** (une transaction = une ligne)
- Les noms de produits sont **cohérents** (pas de variantes orthographiques)
- La **densité** est raisonnable (matrices trop denses = tout est corrélé)


In [ ]:
# ── Contrôle qualité ──────────────────────────────────────────────────────────
print("CONTRÔLE QUALITÉ")
print("-" * 40)
print(f"Doublons de transaction_id : {basket.index.duplicated().sum()}")
print(f"Valeurs NaN dans le basket : {basket.isnull().sum().sum()}")
print(f"Produits avec 0 occurrence : {(basket.sum() == 0).sum()}")

# Fréquence de chaque produit
product_freq = basket.sum().sort_values(ascending=False)
print(f"\nTop 5 produits les plus présents :")
for p, cnt in product_freq.head(5).items():
    print(f"  {p:<30} {cnt:>5} transactions ({cnt/N_TRANSACTIONS:.1%})")

print(f"\nTop 5 produits les moins présents :")
for p, cnt in product_freq.tail(5).items():
    print(f"  {p:<30} {cnt:>5} transactions ({cnt/N_TRANSACTIONS:.1%})")

---
# ⚙️ Section 5 — Algorithme Apriori

## 5.1 Paramétrage et exécution

Le paramètre **`min_support`** est le seuil le plus délicat à fixer :
- Trop **élevé** → on manque des règles intéressantes sur les produits rares
- Trop **faible** → explosion combinatoire, des milliers de règles inutilisables

<div style="background: #e8f4f8; border-left: 4px solid #4a9eff;
     padding: 12px 18px; border-radius: 6px; margin: 10px 0;">
<b>🎯 Point de départ recommandé :</b> min_support = 0.01 (1%)
→ Un produit doit apparaître dans au moins 30 transactions (sur 3 000) pour être considéré.
</div>


In [ ]:
# ── Calcul des itemsets fréquents ─────────────────────────────────────────────
MIN_SUPPORT = 0.01   # ← Paramètre clé — essayez 0.005, 0.02, 0.05 !

frequent_itemsets = apriori(
    basket,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=4           # itemsets de 1 à 4 produits max
)

frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets.sort_values(
    ['length', 'support'], ascending=[True, False])

print(f"✅ {len(frequent_itemsets)} itemsets fréquents (support ≥ {MIN_SUPPORT:.0%})")
print()

# Répartition par taille
print("Répartition par taille d'itemset :")
for size, grp in frequent_itemsets.groupby('length'):
    label = {1: "singleton", 2: "paires", 3: "triplets", 4: "quadruplets"}
    print(f"  Taille {size} ({label.get(size,''):<12}) : {len(grp):>4} itemsets")

print()
print("Top 10 itemsets les plus fréquents (toutes tailles) :")
display_if = frequent_itemsets.copy()
display_if['itemsets'] = display_if['itemsets'].apply(lambda x: ', '.join(sorted(list(x))))
display_if[['itemsets', 'length', 'support']].head(10)

## 5.2 Impact du min_support — Visualisation

In [ ]:
# ── Sensibilité au paramètre min_support ─────────────────────────────────────
support_values = [0.005, 0.008, 0.01, 0.015, 0.02, 0.03, 0.05]
counts_by_support = []

for sup in support_values:
    fi = apriori(basket, min_support=sup, use_colnames=True, max_len=4)
    fi['length'] = fi['itemsets'].apply(len)
    counts = fi.groupby('length').size().to_dict()
    counts['total'] = len(fi)
    counts['support'] = sup
    counts_by_support.append(counts)

df_sensitivity = pd.DataFrame(counts_by_support).fillna(0).astype(
    {c: int for c in [1, 2, 3, 4, 'total']})

fig, ax = plt.subplots(figsize=(11, 5))
x = range(len(support_values))
width = 0.2

for i, (size, color, label) in enumerate([
    (1, '#4a9eff', 'Singletons'),
    (2, '#6bcb77', 'Paires'),
    (3, '#ffd93d', 'Triplets'),
    (4, '#ff6b6b', 'Quadruplets')
]):
    vals = [counts_by_support[j].get(size, 0) for j in range(len(support_values))]
    bars = ax.bar([xi + i * width for xi in x], vals,
                  width=width, label=label, color=color, alpha=0.85)

ax.set_xticks([xi + 1.5 * width for xi in x])
ax.set_xticklabels([f"{s:.1%}" for s in support_values])
ax.set_xlabel("min_support")
ax.set_ylabel("Nombre d'itemsets fréquents")
ax.set_title("Impact du paramètre min_support sur le nombre d'itemsets", pad=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Annotation du seuil utilisé
ax.axvline(x=support_values.index(0.01) + 1.5 * width,
           color='white', linestyle=':', linewidth=2, alpha=0.6,
           label='Seuil utilisé (1%)')

plt.tight_layout()
plt.show()

print("\n💡 Observation : en dessous de 0.5%, le nombre de règles explose")
print("   et devient ingérable en contexte pédagogique.")

---
# 🔗 Section 6 — Génération des Règles d'Association

## 6.1 Génération avec mlxtend


In [ ]:
# ── Génération des règles ─────────────────────────────────────────────────────
rules_raw = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.2
)

print(f"Règles brutes générées : {len(rules_raw)}")

# ── Filtrage multi-critères ───────────────────────────────────────────────────
MIN_CONFIDENCE = 0.40
MIN_LIFT       = 1.50
MIN_SUPPORT_R  = 0.01

rules = rules_raw[
    (rules_raw['confidence'] >= MIN_CONFIDENCE) &
    (rules_raw['lift']       >= MIN_LIFT)        &
    (rules_raw['support']    >= MIN_SUPPORT_R)
].sort_values('lift', ascending=False).reset_index(drop=True)

print(f"Règles après filtrage      : {len(rules)}")
print(f"  (conf ≥ {MIN_CONFIDENCE:.0%} | lift ≥ {MIN_LIFT} | support ≥ {MIN_SUPPORT_R:.0%})")
print()

# Affichage lisible
rules_display = rules.copy()
rules_display['SI (antécédent)']    = rules_display['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
rules_display['ALORS (conséquent)'] = rules_display['consequents'].apply(lambda x: ', '.join(sorted(list(x))))
rules_display[['SI (antécédent)', 'ALORS (conséquent)',
               'support', 'confidence', 'lift']].head(15)

## 6.2 Statistiques sur les règles extraites

In [ ]:
# ── Distribution des métriques ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = [
    ('support',    '#4a9eff', 'Support',    'Fréquence dans les transactions'),
    ('confidence', '#6bcb77', 'Confiance',  'Force directionnelle'),
    ('lift',       '#ffd93d', 'Lift',       'Corrélation vs. hasard'),
]

for ax, (col, color, title, subtitle) in zip(axes, metrics):
    ax.hist(rules[col], bins=25, color=color, edgecolor='white',
            linewidth=0.4, alpha=0.85)
    ax.axvline(rules[col].mean(), color='white', linestyle='--',
               linewidth=1.5, label=f'Moy. = {rules[col].mean():.2f}')
    ax.axvline(rules[col].median(), color='#ff6b6b', linestyle=':',
               linewidth=1.5, label=f'Méd. = {rules[col].median():.2f}')
    ax.set_title(f'{title}\n{subtitle}', fontsize=10, pad=8)
    ax.set_xlabel(col.capitalize())
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Distribution des métriques sur les règles filtrées", fontsize=12)
plt.tight_layout()
plt.show()

print(f"\nSynthèse des {len(rules)} règles filtrées :")
print(rules[['support', 'confidence', 'lift']].describe().round(3).to_string())

---
# 📊 Section 7 — Visualisations

## 7.1 Heatmap principale — Support × Confiance × Lift


In [ ]:
# ── Heatmap principale ────────────────────────────────────────────────────────
top_rules = rules.head(25).copy()
top_rules['ante_str'] = top_rules['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
top_rules['cons_str'] = top_rules['consequents'].apply(lambda x: ', '.join(sorted(list(x))))

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# ── Scatter : Support × Confiance, couleur = Lift ────────────────
ax1 = axes[0]
sc = ax1.scatter(
    top_rules['support'], top_rules['confidence'],
    c=top_rules['lift'], s=top_rules['lift'] ** 2.2 * 15,
    cmap='RdYlGn', alpha=0.85, edgecolors='white', linewidths=0.5,
    vmin=1.0, vmax=top_rules['lift'].max()
)
cbar = plt.colorbar(sc, ax=ax1)
cbar.set_label('Lift', color='white', fontsize=10)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

ax1.axhline(y=0.60, color='#f0c040', linestyle='--', lw=1.2, alpha=0.7, label='Confiance 60%')
ax1.axvline(x=0.02, color='#40c0f0', linestyle='--', lw=1.2, alpha=0.7, label='Support 2%')

for _, row in top_rules.head(6).iterrows():
    ax1.annotate(
        f"{row['ante_str']}\n→ {row['cons_str']}",
        (row['support'], row['confidence']),
        fontsize=6, color='white', alpha=0.9,
        xytext=(8, 5), textcoords='offset points',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='#2a3a50', alpha=0.7)
    )

ax1.set_xlabel('Support', fontsize=11)
ax1.set_ylabel('Confiance', fontsize=11)
ax1.set_title('Support × Confiance\n(taille & couleur = Lift)', fontsize=11, pad=12)
ax1.legend(fontsize=8, facecolor='#1a2535', labelcolor='white')

# ── Bar chart : Top 15 par Lift ───────────────────────────────────
ax2 = axes[1]
top15 = top_rules.head(15).sort_values('lift')
colors_bar = plt.cm.RdYlGn(
    (top15['lift'] - top15['lift'].min()) /
    (top15['lift'].max() - top15['lift'].min() + 0.001)
)
bars = ax2.barh(range(len(top15)), top15['lift'],
                color=colors_bar, edgecolor='white', linewidth=0.4, height=0.7)
ax2.axvline(x=1.0, color='#ff4444', linestyle='--', lw=1.5, label='Lift = 1 (hasard)')
for bar, row in zip(bars, top15.itertuples()):
    ax2.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             f'{row.lift:.1f}× | {row.confidence:.0%}', va='center', fontsize=7.5)

ax2.set_yticks(range(len(top15)))
ax2.set_yticklabels(
    [f"{r.ante_str} → {r.cons_str}" for r in top15.itertuples()],
    fontsize=7
)
ax2.set_xlabel('Lift')
ax2.set_title('Top 15 règles par Lift', fontsize=11, pad=12)
ax2.legend(fontsize=8, facecolor='#1a2535', labelcolor='white')
ax2.set_xlim(0, top15['lift'].max() * 1.25)

fig.suptitle(
    f'SwissShop — Analyse des Règles d\'Association\n'
    f'{len(rules)} règles | {N_TRANSACTIONS:,} transactions | '
    f'min_support={MIN_SUPPORT_R:.0%} | min_confidence={MIN_CONFIDENCE:.0%}',
    fontsize=12, y=1.02
)
plt.tight_layout(pad=2.0)
plt.show()

## 7.2 Graphe réseau des associations

In [ ]:
# ── Réseau d'associations ─────────────────────────────────────────────────────
G = nx.DiGraph()

for _, row in rules.head(20).iterrows():
    ante = ', '.join(sorted(list(row['antecedents'])))
    cons = ', '.join(sorted(list(row['consequents'])))
    G.add_edge(ante, cons, weight=row['lift'],
               confidence=row['confidence'], support=row['support'])

fig, ax = plt.subplots(figsize=(16, 11))
pos = nx.spring_layout(G, k=3.5, seed=42, iterations=60)

# Couleurs par catégorie
color_map_cat = {
    "Informatique": "#4a9eff", "Téléphonie": "#ff6b6b",
    "Audio": "#ffd93d",        "Bureau": "#6bcb77", "Accessoires": "#c77dff"
}
def get_node_color(node):
    for cat, prods in CATALOGUE.items():
        if any(p in node for p in prods):
            return color_map_cat.get(cat, "#aaaaaa")
    return "#aaaaaa"

node_colors = [get_node_color(n) for n in G.nodes()]
degrees = dict(G.degree())
node_sizes = [max(900, degrees[n] * 700) for n in G.nodes()]

edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
edge_norm = [(w - min(edge_weights)) / (max(edge_weights) - min(edge_weights) + 0.001)
             for w in edge_weights]

nx.draw_networkx_nodes(G, pos, node_size=node_sizes,
                       node_color=node_colors, alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7,
                        font_color='white', font_weight='bold', ax=ax)
nx.draw_networkx_edges(G, pos, width=[1 + w * 3 for w in edge_norm],
                       edge_color=[plt.cm.YlOrRd(w) for w in edge_norm],
                       arrows=True, arrowsize=15,
                       connectionstyle='arc3,rad=0.1', ax=ax)

legend_handles = [mpatches.Patch(color=c, label=cat)
                  for cat, c in color_map_cat.items()]
ax.legend(handles=legend_handles, loc='lower left',
          facecolor='#1a2535', labelcolor='white', fontsize=9)
ax.set_title('Réseau des règles d\'association\n'
             'Épaisseur/couleur = Lift | Taille des nœuds = Connectivité',
             fontsize=12, pad=15)
ax.axis('off')
plt.tight_layout()
plt.show()

## 7.3 Matrice de chaleur — Produits les plus associés

In [ ]:
# ── Heatmap de co-occurrence produit × produit ────────────────────────────────
# Sélectionner les 15 produits les plus fréquents
top15_products = product_freq.head(15).index.tolist()
basket_top15 = basket[top15_products]

# Matrice de co-occurrence normalisée
cooc = basket_top15.T.dot(basket_top15.astype(int))
cooc_norm = cooc / N_TRANSACTIONS  # → proportion de co-occurrence

np.fill_diagonal(cooc_norm.values, 0)  # exclure la diagonale (produit avec lui-même)

fig, ax = plt.subplots(figsize=(13, 10))
mask = cooc_norm == 0

sns.heatmap(
    cooc_norm,
    ax=ax,
    cmap='YlOrRd',
    annot=True,
    fmt='.2%',
    linewidths=0.3,
    linecolor='#1a2535',
    annot_kws={'size': 7},
    cbar_kws={'label': 'Co-occurrence (% transactions)'}
)

ax.set_title('Matrice de co-occurrence — Top 15 produits\n'
             '(% de transactions contenant les deux produits simultanément)',
             fontsize=11, pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

print("\n💡 Lecture : la cellule (Imprimante laser, Cartouche toner)")
print("   indique le % de transactions où ces deux produits sont achetés ensemble.")

---
# 💼 Section 8 — Traduction en Règles Business

## 8.1 Rapport automatique des recommandations

<div style="background: #f0f8e8; border-left: 4px solid #6bcb77;
     padding: 12px 18px; border-radius: 6px; margin: 10px 0; color: #1a3a1a;">
<b>🎯 Objectif :</b> Transformer les résultats statistiques en <b>actions concrètes</b>
lisibles par un directeur commercial, sans prérequis en data science.
</div>


In [ ]:
# ── Rapport business automatique ─────────────────────────────────────────────
def generate_business_report(rules, top_n=12):
    print("=" * 72)
    print("  SWISSSHOP — RAPPORT DES RÈGLES BUSINESS")
    print("  Généré par : Analyse Apriori (CRISP-DM)")
    print(f"  Basé sur  : {N_TRANSACTIONS:,} transactions | 2024")
    print("=" * 72)

    # Catégorisation par niveau d'action
    critiques   = rules[(rules['lift'] > 5)  & (rules['confidence'] > 0.55)]
    fortes      = rules[(rules['lift'].between(3, 5)) & (rules['confidence'] > 0.45)]
    moderees    = rules[(rules['lift'].between(1.5, 3)) & (rules['confidence'] > 0.40)]

    for label, subset, icon, action in [
        ("CROSS-SELL PRIORITAIRE",     critiques, "🔴", "Proposer systématiquement à la caisse / dans l'e-mail de confirmation"),
        ("BUNDLE RECOMMANDÉ",          fortes,    "🟡", "Créer une offre groupée avec remise 5-10%"),
        ("PLACEMENT RAYON / VITRINE",  moderees,  "🟢", "Rapprocher en rayon ou dans la navigation web"),
    ]:
        if len(subset) == 0:
            continue
        print(f"\n{'─'*72}")
        print(f"  {icon} {label} ({len(subset)} règle(s))")
        print(f"  ACTION : {action}")
        print(f"{'─'*72}")

        for _, row in subset.head(4).iterrows():
            ante = ', '.join(sorted(list(row['antecedents'])))
            cons = ', '.join(sorted(list(row['consequents'])))
            pop = int(row['support'] * N_TRANSACTIONS)
            print(f"\n  SI  : {ante}")
            print(f"  →   : {cons}")
            print(f"  Stats : Confiance {row['confidence']:.0%} | Lift {row['lift']:.1f}× | "
                  f"~{pop} transactions/an concernées")

    print(f"\n{'═'*72}")
    print("  AVERTISSEMENT : Ces règles reflètent des CO-OCCURRENCES statistiques.")
    print("  Elles ne prouvent pas de CAUSALITÉ. Toute règle doit être validée")
    print("  par un expert métier avant opérationnalisation.")
    print(f"{'═'*72}")

generate_business_report(rules)

## 8.2 Export CSV des règles pour Power BI / Excel

In [ ]:
# ── Export des règles pour Power BI ──────────────────────────────────────────
rules_export = rules.copy()
rules_export['antecedents'] = rules_export['antecedents'].apply(
    lambda x: ', '.join(sorted(list(x))))
rules_export['consequents'] = rules_export['consequents'].apply(
    lambda x: ', '.join(sorted(list(x))))
rules_export['support']    = rules_export['support'].round(4)
rules_export['confidence'] = rules_export['confidence'].round(4)
rules_export['lift']       = rules_export['lift'].round(3)

# Ajout d'une colonne "priorité"
def priorite(row):
    if row['lift'] > 5 and row['confidence'] > 0.55: return "1_Critique"
    if row['lift'] > 3: return "2_Forte"
    return "3_Modérée"

rules_export['priorite'] = rules_export.apply(priorite, axis=1)

# Export
rules_export.to_csv('swissshop_regles_business.csv', index=False, encoding='utf-8-sig')
df.to_csv('swissshop_transactions.csv', index=False, encoding='utf-8-sig')

print("✅ Fichiers exportés :")
print("   • swissshop_regles_business.csv  → import direct dans Power BI")
print("   • swissshop_transactions.csv     → dataset brut pour analyses BI")
print()
print(f"Aperçu des {len(rules_export)} règles exportées :")
rules_export[['antecedents', 'consequents', 'support',
              'confidence', 'lift', 'priorite']].head(10)

---
# 🏋️ Section 9 — Exercices Pratiques

<div style="background: #f0e8ff; border-left: 4px solid #c77dff;
     padding: 15px 20px; border-radius: 6px; margin: 10px 0; color: #2a1a3a;">
<b>📌 Instructions :</b> Complétez les cellules marquées <code># VOTRE CODE ICI</code>.
Chaque exercice est indépendant. Les corrections sont disponibles auprès de l'enseignant.
</div>

## Exercice 1 — Analyse par catégorie de produits

**Objectif :** Relancer l'analyse Apriori **uniquement sur la catégorie "Téléphonie"**
et comparer les règles obtenues avec l'analyse globale.


In [ ]:
# ── Exercice 1 — Analyse par catégorie ───────────────────────────────────────
# 1. Filtrer df pour ne garder que les produits de catégorie "Téléphonie"
# 2. Construire le basket filtré
# 3. Lancer Apriori avec min_support=0.02
# 4. Afficher les 5 meilleures règles

# VOTRE CODE ICI ──────────────────────────────────────────────────────────────

# df_tel = df[df['category'] == ???]
# basket_tel = ...
# fi_tel = apriori(...)
# rules_tel = association_rules(...)
# ...
# ─────────────────────────────────────────────────────────────────────────────

# Résultat attendu : 3-8 règles centrées sur Smartphone/Coque/Protection écran

---
## Exercice 2 — Analyse saisonnière

**Objectif :** Comparer les règles du **T4 2024 (oct-déc)** avec le reste de l'année.
*Hypothèse : les achats de Noël génèrent-ils des patterns différents ?*


In [ ]:
# ── Exercice 2 — Saisonnalité ─────────────────────────────────────────────────
# 1. Créer deux sous-datasets : df_q4 (oct-déc) et df_autres
# 2. Construire un basket pour chaque période
# 3. Générer les règles pour chaque période
# 4. Trouver les règles UNIQUEMENT présentes en Q4 (pas dans les autres mois)

# Indice : df['date'].dt.month pour extraire le mois

# VOTRE CODE ICI ──────────────────────────────────────────────────────────────

# df_q4     = df[df['date'].dt.month >= ???]
# df_autres = df[df['date'].dt.month < ???]
# ...
# ─────────────────────────────────────────────────────────────────────────────

---
## Exercice 3 — Tuning du modèle

**Objectif :** Trouver le couple **(min_support, min_confidence)** optimal
qui produit entre 50 et 100 règles avec un lift moyen ≥ 3.0.

Créez une **grille de recherche** (grid search) et affichez les résultats sous forme de tableau.


In [ ]:
# ── Exercice 3 — Grid Search des hyperparamètres ─────────────────────────────
support_grid    = [0.005, 0.01, 0.015, 0.02]
confidence_grid = [0.30, 0.40, 0.50, 0.60]

results_grid = []

# VOTRE CODE ICI ──────────────────────────────────────────────────────────────
# Pour chaque combinaison (sup, conf) :
# - Lancer apriori + association_rules
# - Enregistrer : nb_rules, lift_moyen, conf_moyenne
# - Identifier la combinaison qui respecte les critères

for sup in support_grid:
    for conf in confidence_grid:
        pass  # ← Remplacer par votre code
        # fi = apriori(...)
        # r  = association_rules(...)
        # results_grid.append({...})

# pd.DataFrame(results_grid)
# ─────────────────────────────────────────────────────────────────────────────

---
## Exercice 4 *(Bonus)* — Interprétation critique

**Objectif :** Identifier **3 règles "triviales"** dans les résultats
(des règles statistiquement valides mais business inutiles car évidentes)
et expliquer pourquoi elles doivent être filtrées.

*Exemple de règle triviale : `{Cartouche toner} → {Imprimante laser}` — tout le monde sait ça.*


In [ ]:
# ── Exercice 4 — Identification des règles triviales ─────────────────────────
# Examinez les règles et identifiez 3 règles "trop évidentes"
# Justifiez dans les commentaires pourquoi elles n'apportent pas de valeur

# VOTRE CODE ICI ──────────────────────────────────────────────────────────────

# règles_triviales = rules[rules['antecedents'].apply(lambda x: ???)]
# Expliquez : pourquoi sont-elles triviales ? Comment les filtrer automatiquement ?

# ─────────────────────────────────────────────────────────────────────────────
# RÉFLEXION (répondez en commentaire) :
# 1. Comment distinguer algorithmiquement une règle triviale d'une règle utile ?
# 2. Quel rôle joue l'expert métier dans cette validation ?
# 3. Citez un exemple de règle qui semblerait triviale mais serait en réalité utile.

---
# 🎓 Section 10 — Synthèse & Points Clés

## 10.1 Ce que vous avez appris

| Concept | Méthode | Résultat |
|---------|---------|---------|
| Market Basket Analysis | Reformulation du problème business | Question claire et mesurable |
| Support | Fréquence de co-occurrence | Seuil de pertinence statistique |
| Confiance | Probabilité conditionnelle P(B\|A) | Force directionnelle de la règle |
| Lift | Rapport confiance / support marginal | Pertinence vs. hasard |
| Apriori | Anti-monotonicité + pruning | Efficacité sur grands catalogues |
| Visualisation | Scatter, réseau, heatmap | Communication des résultats |
| Traduction business | SI...ALORS + métriques | Recommandations actionnables |

---

## 10.2 Positionnement dans CRISP-DM

```
✅ Business Understanding  → Objectif : découvrir des patterns de co-achat
✅ Data Understanding      → Exploration du dataset SwissShop
✅ Data Preparation        → Matrice booléenne, nettoyage
✅ Modeling                → Apriori, paramétrage min_support/confidence/lift
✅ Evaluation              → Filtrage des règles, validation business
⬜ Deployment              → Export CSV → Power BI (exercice suivant)
```

---

## 10.3 Limites importantes à retenir

<div style="background: #fef3e2; border-left: 4px solid #e6a817;
     padding: 15px 20px; border-radius: 6px; margin: 15px 0;">

**1. Corrélation ≠ Causalité**
Apriori détecte des co-occurrences statistiques, pas des relations causales.
`{Couches} → {Bière}` (Walmart) était une corrélation accidentelle, pas une cause.

**2. Biais de popularité**
Les produits très vendus apparaissent dans beaucoup de règles sans réelle association.
→ Le lift corrige partiellement ce biais, mais pas totalement.

**3. Saisonnalité ignorée**
Une règle vraie en décembre peut être fausse en juillet.
→ Segmentez par période avant de déployer.

**4. Les règles vieillissent**
Un modèle entraîné sur 2024 peut être obsolète en 2026 si le catalogue change.
→ Réentraînez régulièrement (mensuel minimum pour du retail).

**5. Scalabilité**
Apriori classique est lent sur des millions de transactions.
→ Pour les grands volumes : FP-Growth (plus rapide, même résultats).
</div>

---

## 10.4 Pour aller plus loin

| Étape suivante | Outil | Apport |
|----------------|-------|--------|
| Segmentation clients | K-Means | Règles par segment (VIP vs. standard) |
| Prédiction de churn | XGBoost | Identifier les clients à risque |
| Recommandation temps réel | Collaborative Filtering | Personnalisation e-commerce |
| Visualisation BI | Power BI + DAX | Dashboard exécutif |
| Mise en production | FastAPI + Docker | API de recommandation live |

---

## 10.5 Questions d'examen type

1. Définissez le Lift et expliquez pourquoi une règle avec confidence=0.95 peut être inutile.
2. Pourquoi l'algorithme Apriori est-il plus efficace qu'une énumération complète ?
3. Vous obtenez 50 000 règles avec min_support=0.001. Quels risques cela pose-t-il ?
4. Quelle différence entre une règle d'association et un modèle prédictif ?
5. Un directeur commercial vous demande d'"automatiser" les règles trouvées. Que répondez-vous ?

---

*Notebook réalisé dans le cadre du cours Business Intelligence — HES-SO Valais*
